In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus(n_gpus=1)

import scanpy as sc
from essential.ode import ODEstimator
from essential.utils import compute_topk_precision_metrics
import matplotlib.pyplot as plt

In [ ]:
from configs.tanhdynamic import get_config

config = get_config()
print(config)

In [ ]:
adata = sc.read_h5ad(config.processing.adata_path)
if config.processing.rt_bc != "all":
    adata = adata[adata.obs["rt_bc"] == config.processing.rt_bc].copy()
if config.processing.consolidated_cluster != "all":
    adata = adata[
        adata.obs["consolidated_cluster"] == config.processing.consolidated_cluster
    ].copy()
sc.pp.filter_genes(adata, min_cells=10)

adata

In [ ]:
adata

In [ ]:
# model fitting
ODEstimator.process_data(adata, latent_obsm_key=config.processing.latent_obsm_key)
estimator = ODEstimator(adata, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())

In [ ]:
estimator.step_history_df["loss"].plot()
plt.yscale("log")

In [ ]:
estimator.epoch_history_df["val_loss"].plot()
plt.yscale("log")

In [ ]:
estimator.topk_history_df

In [ ]:
a_mat = estimator.get_interaction_matrix()
processed_a_mat = ODEstimator.process_interaction_matrix(a_mat, return_square=False, delta=0.1)

In [ ]:
topk_precision_df = compute_topk_precision_metrics(processed_a_mat, "model")

In [ ]:
topk_precision_df.query("type == 'offdiag'").groupby("direction")["is_evidence"].sum()